# 06 Pooling And Representation

Reviewer-facing pooling sensitivity and representation-evidence status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Scope And Contracts

This notebook has two reviewer-facing analyses:

- Pooling sensitivity from checksum-verified frozen OOF slice probabilities.
- Optional representation embedding extraction and probe/CKA analysis from frozen final_ema checkpoints.

Representation evidence remains claim-limited: even when probes run, wording must be "suggestive branch-specific information", not proven morphology-rhythm disentanglement.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

THRESHOLD = 0.5
RUN_POOLING_SENSITIVITY = True
RUN_REPRESENTATION_EMBEDDING_EXTRACTION = False
RUN_REPRESENTATION_PROBING = False
REPRESENTATION_ONLY_FOLDS = ''  # Example: '1,2' for interrupted Colab sessions; empty means all/cache assemble.
REPRESENTATION_BATCH_SIZE = 64
REPRESENTATION_NUM_WORKERS = 0
OOF_STEM = 'oof_final_ema'

revision_root = Path('reports/revision')
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
record_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
slice_path = revision_root / 'predictions' / f'{OOF_STEM}_slice_predictions.npz'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'

required_inputs = {
    'oof_final_ema_freeze_manifest': freeze_path,
    'frozen_final_ema_predictions': record_path,
    'frozen_final_ema_slice_predictions': slice_path,
    'calibration_ci_final_ema': calibration_path,
}
for name, path in required_inputs.items():
    print(f'{name:34s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Notebook 02/03 final_ema artifacts are required before notebook 06: ' + '; '.join(missing))

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before pooling sensitivity.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 06 requires checkpoint_kind=final_ema in the freeze manifest.')

frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}
for artifact_path in [record_path, slice_path]:
    rel = artifact_path.as_posix()
    if rel not in frozen_artifacts:
        raise ValueError(f'Freeze manifest does not include {rel}')
    current_sha = sha256_file(artifact_path)
    if current_sha != frozen_artifacts[rel]['sha256']:
        raise RuntimeError(f'Frozen artifact checksum changed: {rel}')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'oof_stem': OOF_STEM,
    'threshold': THRESHOLD,
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'checkpoint_kind': freeze.get('checkpoint_kind'),
    'allowed_execution': {
        'pooling_sensitivity': RUN_POOLING_SENSITIVITY,
        'representation_embedding_extraction': RUN_REPRESENTATION_EMBEDDING_EXTRACTION,
        'representation_probing': RUN_REPRESENTATION_PROBING,
        'model_training': False,
        'model_inference': bool(RUN_REPRESENTATION_EMBEDDING_EXTRACTION),
    },
    'representation_controls': {
        'only_folds': REPRESENTATION_ONLY_FOLDS,
        'batch_size': REPRESENTATION_BATCH_SIZE,
        'num_workers': REPRESENTATION_NUM_WORKERS,
    },
}
save_json(revision_root / 'manifests' / 'pooling_representation_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'pooling_representation_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 06 to the tracked revision plan and claim map. It separates supported pooling analysis from still-blocked representation evidence.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_POOL', 'EXP_REPR'])]
relevant_claims = claims[claims['claim_id'].isin(['C04', 'C05'])]
relevant_tasks = tasks[tasks['id'].isin(['B1', 'B3'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'pooling_representation_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'pooling_representation_plan_alignment.json')


## Run Pooling Sensitivity

This uses only checksum-verified frozen OOF slice probabilities. It performs no model inference, no training, and no threshold tuning.


In [ ]:
pooling_command = (
    f'python -u scripts/revision/07_pooling_sensitivity.py '
    f'--freeze-manifest {freeze_path} '
    f'--record-file {record_path} '
    f'--slice-file {slice_path} '
    f'--expected-checkpoint-kind final_ema '
    f'--threshold {THRESHOLD}'
)
if RUN_POOLING_SENSITIVITY:
    run(pooling_command, log_path='reports/revision/logs/oof_final_ema_pooling_sensitivity.log')
else:
    print('Pooling sensitivity disabled. Planned command:', pooling_command)


## Inspect Pooling Results And Decide Q=3 Wording

The decision artifact records whether Q=3 is best, competitive, or should be described only as the frozen/default aggregation setting.


In [ ]:
pooling_csv = revision_root / 'metrics' / 'pooling_sensitivity.csv'
pooling_json = revision_root / 'metrics' / 'pooling_sensitivity.json'
if not pooling_csv.exists() or not pooling_json.exists():
    raise FileNotFoundError('Pooling sensitivity outputs are missing; run the previous cell first.')

pooling_df = pd.read_csv(pooling_csv)
expected_methods = {'mean', 'power_mean_q2', 'power_mean_q3', 'power_mean_q4', 'power_mean_q8', 'max'}
observed_methods = set(pooling_df['pooling'].astype(str))
missing_methods = sorted(expected_methods - observed_methods)
if missing_methods:
    raise ValueError('Pooling sensitivity is incomplete; missing methods: ' + ', '.join(missing_methods))

metric_priority = ['roc_auc_macro', 'pr_auc_macro', 'f1_macro', 'recall_macro', 'specificity_macro']
q3 = pooling_df[pooling_df['pooling'] == 'power_mean_q3'].iloc[0].to_dict()
rank_summary = {}
for metric in metric_priority:
    ordered = pooling_df.sort_values(metric, ascending=False).reset_index(drop=True)
    rank = int(ordered.index[ordered['pooling'] == 'power_mean_q3'][0] + 1)
    best = ordered.iloc[0].to_dict()
    rank_summary[metric] = {
        'q3_rank': rank,
        'q3_value': float(q3[metric]),
        'best_pooling': best['pooling'],
        'best_value': float(best[metric]),
    }

q3_best_metrics = [metric for metric, row in rank_summary.items() if row['q3_rank'] == 1]
if q3_best_metrics:
    q3_decision = 'q3_supported_for_metrics_' + '_'.join(q3_best_metrics)
    safe_wording = 'Q=3 is supported for the listed metrics but should still be reported with the full sensitivity table.'
else:
    q3_decision = 'q3_not_best_present_as_frozen_default_or_tradeoff'
    safe_wording = 'Do not claim Q=3 is optimal; present it as the frozen/default aggregation and report the better pooling alternatives.'

decision = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_pooling_csv': str(pooling_csv),
    'source_pooling_json': str(pooling_json),
    'source_pooling_csv_sha256': sha256_file(pooling_csv),
    'threshold': THRESHOLD,
    'expected_methods': sorted(expected_methods),
    'rank_summary': rank_summary,
    'q3_decision': q3_decision,
    'safe_wording': safe_wording,
}
save_json(revision_root / 'metrics' / 'pooling_decision_summary.json', decision)
pooling_table = revision_root / 'tables' / 'table_pooling_sensitivity.csv'
pooling_df.to_csv(pooling_table, index=False)
print('Wrote:', revision_root / 'metrics' / 'pooling_decision_summary.json')
print('Wrote:', pooling_table)
display(pooling_df)
print('Q=3 decision:', q3_decision)
print('Safe wording:', safe_wording)


## Representation Embeddings And Probe/CKA

Optional representation evidence now has an implemented extraction hook and probe runner. Keep both flags `False` unless you intentionally want to run model inference/probing. The extractor writes fold-level caches so interrupted Colab sessions can resume with `REPRESENTATION_ONLY_FOLDS`.

Safe claim boundary: results may support branch-specific information patterns only; they do not prove strict morphology-rhythm disentanglement.


In [ ]:
embedding_path = revision_root / 'predictions' / 'representation_embeddings_final_ema.npz'
run_manifest_path = revision_root / 'manifests' / 'oof_final_ema_prediction_run_manifest.json'
embedding_manifest = revision_root / 'manifests' / 'representation_embedding_manifest.json'
probe_summary = revision_root / 'metrics' / 'representation_probe_summary.json'
probe_table = revision_root / 'tables' / 'table_representation_probe.csv'
cka_table = revision_root / 'tables' / 'table_representation_cka.csv'
probe_manifest = revision_root / 'manifests' / 'representation_probe_manifest.json'

extract_command = (
    'python -u scripts/revision/22_extract_representations.py '
    '--checkpoint-kind final_ema '
    f'--oof-predictions {record_path} '
    f'--freeze-manifest {freeze_path} '
    f'--oof-run-manifest {run_manifest_path} '
    f'--out-embedding {embedding_path} '
    f'--out-manifest {embedding_manifest} '
    f'--batch-size {REPRESENTATION_BATCH_SIZE} '
    f'--num-workers {REPRESENTATION_NUM_WORKERS}'
)
if REPRESENTATION_ONLY_FOLDS.strip():
    extract_command += f' --only-folds "{REPRESENTATION_ONLY_FOLDS}"'

probe_command = (
    'python -u scripts/revision/20_representation_probe.py '
    f'--embedding-npz {embedding_path} '
    f'--out-summary {probe_summary} '
    f'--out-probe-table {probe_table} '
    f'--out-cka-table {cka_table} '
    f'--out-manifest {probe_manifest}'
)

print('Representation extraction command:', extract_command)
print('Representation probe command     :', probe_command)

def ensure_mamba_runtime_for_representation():
    import importlib
    import json as _json

    required_modules = ['mamba_ssm', 'causal_conv1d']
    missing = [module for module in required_modules if importlib.util.find_spec(module) is None]
    if not missing:
        print('Mamba runtime available for representation extraction.')
        return

    print('Mamba runtime missing for representation extraction:', missing)
    installer_nb_path = (REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb') if 'REPO_DIR' in globals() else Path('notebooks/02_predictions_and_external_eval.ipynb')
    if not installer_nb_path.exists():
        raise FileNotFoundError(
            'Missing canonical Mamba installer notebook: ' + str(installer_nb_path) +
            '. Run Notebook 00 bootstrap first, or run Notebook 02 model dependency cell in this same runtime.'
        )
    installer_nb = _json.loads(installer_nb_path.read_text(encoding='utf-8'))
    installer_source = None
    for notebook_cell in installer_nb['cells']:
        if notebook_cell.get('cell_type') != 'code':
            continue
        source = ''.join(notebook_cell.get('source', []))
        if 'INSTALL_MODEL_DEPS = True' in source and 'AUTO_PIN_TORCH_FOR_MAMBA' in source:
            installer_source = source
            break
    if installer_source is None:
        raise RuntimeError('Could not locate the canonical Mamba installer cell in notebook 02.')
    print('Running canonical Mamba installer from Notebook 02 before representation extraction...')
    exec(compile(installer_source, str(installer_nb_path) + ':model-deps', 'exec'), globals(), globals())
    importlib.invalidate_caches()
    still_missing = [module for module in required_modules if importlib.util.find_spec(module) is None]
    if still_missing:
        raise ImportError(
            'Mamba runtime remains unavailable after installer: ' + ', '.join(still_missing) +
            '. If Colab intentionally restarted after torch pinning, rerun Notebook 06 from Setup.'
        )

if RUN_REPRESENTATION_EMBEDDING_EXTRACTION:
    ensure_mamba_runtime_for_representation()
    if not Path('scripts/revision/22_extract_representations.py').exists():
        raise FileNotFoundError('Missing scripts/revision/22_extract_representations.py')
    run(extract_command, log_path='reports/revision/logs/representation_embedding_extraction.log')
else:
    print('Representation embedding extraction disabled. Set RUN_REPRESENTATION_EMBEDDING_EXTRACTION=True to run it.')

if RUN_REPRESENTATION_PROBING:
    if not embedding_path.exists():
        raise FileNotFoundError('Representation embedding NPZ is missing. Run extraction first or restore it from mirror.')
    run(probe_command, log_path='reports/revision/logs/representation_probe.log')
else:
    print('Representation probing disabled. Set RUN_REPRESENTATION_PROBING=True after embeddings exist.')

representation_outputs = {
    'embedding_npz': embedding_path,
    'embedding_manifest': embedding_manifest,
    'probe_summary': probe_summary,
    'probe_table': probe_table,
    'cka_table': cka_table,
    'probe_manifest': probe_manifest,
}
representation_rows = []
for name, path in representation_outputs.items():
    exists = path.exists()
    representation_rows.append({
        'evidence_name': name,
        'path': str(path),
        'exists': exists,
        'size_bytes': path.stat().st_size if exists else None,
        'status': 'complete' if exists else 'missing',
        'sha256': sha256_file(path) if exists else '',
    })

if probe_summary.exists() and probe_table.exists() and cka_table.exists():
    representation_status = 'complete_probe_available'
    safe_wording = 'Representation probes provide suggestive branch-specific evidence only; do not claim proven disentanglement.'
elif embedding_path.exists():
    representation_status = 'embeddings_ready_probe_not_run'
    safe_wording = 'Embedding artifact exists, but probe/CKA tables are not complete; do not claim representation separation yet.'
elif embedding_manifest.exists():
    manifest_payload = json.loads(embedding_manifest.read_text(encoding='utf-8'))
    representation_status = manifest_payload.get('status', 'partial_or_blocked')
    safe_wording = manifest_payload.get('safe_wording', 'Do not claim demonstrated morphology-rhythm separation.')
else:
    representation_status = 'blocked_missing_embeddings'
    safe_wording = 'Do not claim demonstrated morphology-rhythm separation; run the embedding extractor and probe first.'

repr_df = pd.DataFrame(representation_rows)
repr_table = revision_root / 'tables' / 'table_representation_evidence_status.csv'
repr_json = revision_root / 'metrics' / 'representation_evidence_status.json'
repr_df.to_csv(repr_table, index=False)
save_json(repr_json, {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'run_representation_embedding_extraction': RUN_REPRESENTATION_EMBEDDING_EXTRACTION,
    'run_representation_probing': RUN_REPRESENTATION_PROBING,
    'status': representation_status,
    'safe_wording': safe_wording,
    'extract_command': extract_command,
    'probe_command': probe_command,
    'rows': representation_rows,
})
print('Wrote:', repr_table)
print('Wrote:', repr_json)
print('Representation status:', representation_status)
display(repr_df)


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/pooling_representation_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/pooling_representation_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'pooling_representation_input_contract.json',
    revision_root / 'manifests' / 'pooling_representation_plan_alignment.json',
    revision_root / 'metrics' / 'pooling_sensitivity.csv',
    revision_root / 'metrics' / 'pooling_sensitivity.json',
    revision_root / 'metrics' / 'pooling_decision_summary.json',
    revision_root / 'tables' / 'table_pooling_sensitivity.csv',
    revision_root / 'metrics' / 'representation_evidence_status.json',
    revision_root / 'tables' / 'table_representation_evidence_status.csv',
    revision_root / 'predictions' / 'representation_embeddings_final_ema.npz',
    revision_root / 'manifests' / 'representation_embedding_manifest.json',
    revision_root / 'metrics' / 'representation_probe_summary.json',
    revision_root / 'tables' / 'table_representation_probe.csv',
    revision_root / 'tables' / 'table_representation_cka.csv',
    revision_root / 'manifests' / 'representation_probe_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
print('Notebook 06 complete. Pooling sensitivity can support/revise Q=3 wording. Representation claims require completed embedding/probe artifacts and remain limited to suggestive branch-specific evidence.')
